In [ ]:
!pip install -U loguru torch torchvision ultralytics

In [ ]:
import os
import cv2
import numpy as np
from google.colab import drive
from loguru import logger
import pandas as pd
import shutil
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

In [ ]:
drive.mount('/content/drive')

In [ ]:
BASE_DIR = '/content/drive/MyDrive/molavi/mri'
OUTPUT_DIR = '/content/drive/MyDrive/molavi/rgb_mri'
CLASSIFICATION_CSV = '/content/drive/MyDrive/molavi/classification_df_asli.csv'
BASE_IMAGE_DIR = '/content/drive/MyDrive/molavi/rgb_mri'  
CAT_OUTPUT_DIR = '/content/drive/MyDrive/molavi/catagorized_rgb_mri'  
DATASET_DIR = '/content/drive/MyDrive/molavi/catagorized_rgb_mri'  
YOLO_DATASET_DIR = '/content/drive/MyDrive/molavi/yolo_dataset' 
TEST_FOLDER = '/content/drive/MyDrive/molavi/yolo_dataset/test' 

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def combine_images_to_rgb(t1_path, t2_path, output_path):
    """
    Combine T1 and T2 images into an RGB image where:
      - T1 is the red channel
      - T2 is duplicated for green and blue channels
    """
    logger.info(f"Combining images: T1 ({t1_path}) and T2 ({t2_path})")

    t1_img = cv2.imread(t1_path, cv2.IMREAD_GRAYSCALE)
    t2_img = cv2.imread(t2_path, cv2.IMREAD_GRAYSCALE)

    if t1_img is None:
        logger.error(f"Failed to load T1 image from {t1_path}")
        return
    if t2_img is None:
        logger.error(f"Failed to load T2 image from {t2_path}")
        return

    if t1_img.shape != t2_img.shape:
        logger.warning(f"Skipping {t1_path} and {t2_path}: Sizes do not match (T1: {t1_img.shape}, T2: {t2_img.shape})")
        return

    rgb_img = np.stack((t1_img, t2_img, t2_img), axis=-1)  # T1 as red, T2 as green and blue

    cv2.imwrite(output_path, rgb_img)
    logger.info(f"Saved combined RGB image to: {output_path}")

def process_dataset(BASE_DIR, OUTPUT_DIR):
    """
    Traverse the dataset directory and combine T1 and T2 images into RGB images.
    """
    logger.info(f"Starting dataset processing from base directory: {BASE_DIR}")
    for root, dirs, files in os.walk(BASE_DIR):
        if 'T1_middles_registerd_png' in root:
            t1_dir = root
            t2_dir = root.replace('T1_middles_registerd_png', 'T2_middles_registerd')

            if os.path.exists(t2_dir):
                logger.info(f"Processing directory: {root}")

                rel_path = os.path.relpath(root, BASE_DIR)
                combined_dir = os.path.join(OUTPUT_DIR, rel_path)
                os.makedirs(combined_dir, exist_ok=True)
                logger.info(f"Created output directory: {combined_dir}")

                for t1_file in os.listdir(t1_dir):
                    t1_path = os.path.join(t1_dir, t1_file)
                    t2_path = os.path.join(t2_dir, t1_file)
                    output_path = os.path.join(combined_dir, t1_file)

                    if os.path.exists(output_path):
                        logger.info(f"Skipping {output_path}: File already exists.")
                        continue

                    if os.path.exists(t2_path):
                        combine_images_to_rgb(t1_path, t2_path, output_path)
                    else:
                        logger.warning(f"Missing T2 file for T1 image: {t1_path}")
            else:
                logger.warning(f"T2 directory not found for T1 directory: {t1_dir}")
    logger.info(f"Dataset processing complete. Combined images saved to: {OUTPUT_DIR}")


In [ ]:
process_dataset(BASE_DIR, OUTPUT_DIR)

In [ ]:
df = pd.read_csv(CLASSIFICATION_CSV)

def categorize_image():
    for group in df['Group'].unique():
        group_path = os.path.join(CAT_OUTPUT_DIR, group)
        os.makedirs(group_path, exist_ok=True)

    for _, row in df.iterrows():
        subject = row['Subjects']
        group = row['Group']

        subject_dir = os.path.join(BASE_IMAGE_DIR, subject)
        group_dir = os.path.join(CAT_OUTPUT_DIR, group)

        if os.path.exists(subject_dir):
            for file_name in os.listdir(subject_dir):
                file_path = os.path.join(subject_dir, file_name)
                if os.path.isfile(file_path):
                    shutil.copy(file_path, group_dir)
                    logger.info(f"copied contents of {subject} to {group_dir}")
        else:
            logger.warning(f"Warning: Subject directory {subject_dir} not found.")

    logger.info("Image categorization complete.")


In [ ]:
categorize_image()

In [ ]:
def prepare_yolo_dataset(input_dir, OUTPUT_DIR, train_split=0.8, val_split=0.1):
    """
    Converts categorized dataset into YOLO format with category subdirectories.
    yolo_dataset/
    ├── train/
    │   ├── CN/
    │   │   ├── image1.jpg
    │   │   ├── image2.jpg
    │   ├── MCI/
    │   │   ├── image3.jpg
    │   ├── AD/
    │       ├── image4.jpg
    ├── val/
    ├── test/
    """

    train_dir = os.path.join(OUTPUT_DIR, 'train')
    val_dir = os.path.join(OUTPUT_DIR, 'val')
    test_dir = os.path.join(OUTPUT_DIR, 'test')

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    images = []
    categories = os.listdir(input_dir)
    for category in categories:
        category_dir = os.path.join(input_dir, category)
        if os.path.isdir(category_dir):
            for image in os.listdir(category_dir):
                if image.endswith(('.png', '.jpg', '.jpeg')):
                    images.append((os.path.join(category_dir, image), category))

    train_images, temp_images = train_test_split(images, train_size=train_split, stratify=[img[1] for img in images])
    val_images, test_images = train_test_split(temp_images, test_size=val_split / (1 - train_split), stratify=[img[1] for img in temp_images])

    for img_path, category in train_images:
        category_train_dir = os.path.join(train_dir, category)
        os.makedirs(category_train_dir, exist_ok=True)
        shutil.copy(img_path, os.path.join(category_train_dir, os.path.basename(img_path)))

    for img_path, category in val_images:
        category_val_dir = os.path.join(val_dir, category)
        os.makedirs(category_val_dir, exist_ok=True)
        shutil.copy(img_path, os.path.join(category_val_dir, os.path.basename(img_path)))

    for img_path, category in test_images:
        category_test_dir = os.path.join(test_dir, category)
        os.makedirs(category_test_dir, exist_ok=True)
        shutil.copy(img_path, os.path.join(category_test_dir, os.path.basename(img_path)))

    print(f"Dataset prepared in YOLO format with category subdirectories at {OUTPUT_DIR}")


In [ ]:
prepare_yolo_dataset(DATASET_DIR, YOLO_DATASET_DIR)

In [ ]:

# # This has to be chosen based on the model you want to train
model = YOLO('yolo11x-cls.pt')
# model = YOLO('yolo11s-cls.pt')
# model = YOLO('/content/drive/MyDrive/molavi/yolo_results/weights/best.pt')

model.train(
    data='/content/drive/MyDrive/molavi/yolo_dataset', 
    epochs=100,                                        
    imgsz=224,                                        
    batch=16,                                      
    name='/content/drive/MyDrive/molavi/yolo_results',
)

results = model.val()


In [ ]:
model.train(
    data='/content/drive/MyDrive/molavi/yolo_dataset',
    epochs=50,                                       
    imgsz=224,                  
    batch=16,                   
    name='/content/drive/MyDrive/molavi/yolo_results', 
    resume=True                                        
)

In [ ]:
import os
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

In [ ]:
model_s = YOLO('/content/drive/MyDrive/molavi/yolo_results_s/weights/best.pt')
model_x = YOLO('/content/drive/MyDrive/molavi/yolo_results_x/weights/best.pt')

test_folder = '/content/drive/MyDrive/molavi/yolo_dataset/test'

def evaluate_model(model, test_dir):
    """
    Evaluates a YOLO model on the test dataset organized by class folders.
    Prints class-wise metrics and overall results.
    """
    y_true = []
    y_pred = []
    class_names = list(model.names.values())

    for category in os.listdir(test_dir):
        category_dir = os.path.join(test_dir, category)
        if os.path.isdir(category_dir):

            for image_name in os.listdir(category_dir):
                if image_name.endswith(('.png', '.jpg', '.jpeg')):
                    image_path = os.path.join(category_dir, image_name)

                    result = model(image_path)[0]
                    predicted_class_idx = result.probs.top1 
                    predicted_class = model.names[predicted_class_idx] 

                    y_true.append(category)
                    y_pred.append(predicted_class)

    print(f"\n{'*' * 10} Model Results {'*' * 10}")
    print("Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))

    print("Confusion Matrix:")
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    print(cm)

print("Evaluating YOLO 11 S")
evaluate_model(model_s, test_folder)

print("Evaluating YOLO 11 X")
evaluate_model(model_x, test_folder)
